# IMPORT LIBRARIES

In [ ]:
import nbformat
from nbconvert import PythonExporter

with open("data_generator.ipynb", "r", encoding="utf-8") as f:
    nb = nbformat.read(f, as_version=4)

exporter = PythonExporter()
script, _ = exporter.from_notebook_node(nb)

with open("data_generator.py", "w", encoding="utf-8") as f:
    f.write(script)

In [ ]:
from optimization_model import build_model
import pyomo.environ as pyo
from __future__ import annotations
import json
import os
import copy


from data_generator import generate_instance, save_instance_json


In [ ]:
def load_instance_json(path: str) -> Dict[str, Any]:
    """Load instance from JSON. Restores tuple keys for known fields."""
    with open(path, "r", encoding="utf-8") as f:
        raw = json.load(f)

    # Fields with tuple keys and their arity
    tuple_key_fields = {
        "dur": 2,     # (t,k)
        "dist": 2,    # (i,j)
        "tt": 3,      # (i,j,m)
        "Hkp": 2,     # (k,p)
    }

    def decode_keys(d: Dict[str, Any], field_name: str) -> Dict[Any, Any]:
        arity = tuple_key_fields.get(field_name, None)
        if arity is None:
            return d
        out = {}
        for k, v in d.items():
            parts = k.split("|")
            if len(parts) != arity:
                # leave as string key if unexpected
                out[k] = v
            else:
                out[tuple(parts)] = v
        return out

    data: Dict[str, Any] = dict(raw)
    for fn in tuple_key_fields:
        if fn in data and isinstance(data[fn], dict):
            data[fn] = decode_keys(data[fn], fn)

    return data

In [ ]:
def export_results(model: pyo.ConcreteModel, filepath: str, save_zero: bool = False):
    """
    Export:
      - objective value
      - all variable values
    to a JSON file.

    save_zero = False -> only store non-zero variables (recommended)
    """

    os.makedirs(os.path.dirname(filepath) or ".", exist_ok=True)

    results = {}

    # ------------------------
    # Objective
    # ------------------------
    obj = next(model.component_data_objects(pyo.Objective, active=True))
    results["objective_value"] = float(pyo.value(obj))

    # ------------------------
    # Variables
    # ------------------------
    results["variables"] = {}

    for var in model.component_objects(pyo.Var, active=True):
        var_name = var.name
        results["variables"][var_name] = {}

        for index in var:
            val = pyo.value(var[index])
            if (not save_zero) and (abs(val) < 1e-9):
                continue

            # Convert index to string
            if isinstance(index, tuple):
                key = "|".join(map(str, index))
            else:
                key = str(index)

            results["variables"][var_name][key] = float(val)

    # ------------------------
    # Save JSON
    # ------------------------
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {filepath}")

# IMPORT DATA

In [ ]:
import os
import copy
import traceback
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def solve_and_export(
    data,
    result_path: str,
    solver_name: str = "cplex",
    timelimit: int = 300,
    tee: bool = False,
):
    """
    Build model, solve it, and export results if solver finds a usable solution.

    Returns:
        (ok, solve_info)

    ok = True only if the solve finished with a feasible/optimal solution
    and results were exported.
    """
    m = build_model(data)

    solver = pyo.SolverFactory(solver_name)
    if solver is None:
        raise RuntimeError(f"Solver '{solver_name}' is not available.")

    try:
        solver.options["timelimit"] = timelimit
    except Exception:
        pass

    res = solver.solve(m, tee=tee)

    status = res.solver.status
    term = res.solver.termination_condition

    ok = False

    if status in {SolverStatus.ok, SolverStatus.warning}:
        if term in {
            TerminationCondition.optimal,
            TerminationCondition.locallyOptimal,
            TerminationCondition.feasible,
            TerminationCondition.maxTimeLimit,
        }:
            try:
                obj_val = pyo.value(m.obj)
                _ = float(obj_val)
                export_results(m, result_path)
                ok = True
            except Exception:
                ok = False

    solve_info = {
        "status": str(status),
        "termination_condition": str(term),
        "exported": ok,
        "result_path": result_path if ok else None,
    }

    return ok, solve_info


def run_experiments(
    *,
    seeds,
    style: str = "clustered",
    n_bases: int = 3,
    n_techs: int = 12,
    n_plants: int = 25,
    n_tickets: int = 50,
    H: int = 30,
    X: int = 5,
    modes=("car", "train", "air"),
    solver_name: str = "cplex",
    timelimit: int = 300,
    tee: bool = False,
    instances_dir: str = "instances",
    results_dir: str = "results",
):
    """
    Run batch experiments over multiple seeds.

    For each seed:
      1. generate and save instance
      2. solve economic baseline -> saves results/economic_baseline_seed<seed>.json
      3. solve sustainability-aware -> saves results/sustainability_aware_seed<seed>.json

    No single aggregate JSON with all results is created.
    """
    ensure_dir(instances_dir)
    ensure_dir(results_dir)

    summary = []

    for seed in seeds:
        print(f"\n{'=' * 70}")
        print(f"Running experiment for seed {seed}")
        print(f"{'=' * 70}")

        try:
            # -------------------------
            # Generate instance
            # -------------------------
            inst = generate_instance(
                style=style,
                seed=seed,
                n_bases=n_bases,
                n_techs=n_techs,
                n_plants=n_plants,
                n_tickets=n_tickets,
                H=H,
                X=X,
                modes=modes,
            )

            instance_path = os.path.join(instances_dir, f"demo_{style}_seed{seed}.json")
            save_instance_json(inst, instance_path)
            print(f"Saved instance: {instance_path}")

            seed_record = {
                "seed": seed,
                "instance_path": instance_path,
                "economic": None,
                "sustainability": None,
            }

            # -------------------------
            # Scenario A: Economic baseline
            # -------------------------
            data_econ = copy.deepcopy(inst)
            data_econ["beta"] = 0.0
            data_econ["eta"] = 0.0
            data_econ["delta"] = 0.0

            econ_result_path = os.path.join(
                results_dir, f"economic_baseline_seed{seed}.json"
            )

            try:
                ok_econ, info_econ = solve_and_export(
                    data=data_econ,
                    result_path=econ_result_path,
                    solver_name=solver_name,
                    timelimit=timelimit,
                    tee=tee,
                )
                seed_record["economic"] = info_econ

                if ok_econ:
                    print(f"Seed {seed}: economic baseline saved to {econ_result_path}")
                else:
                    print(
                        f"Seed {seed}: economic baseline infeasible/failed "
                        f"(status={info_econ['status']}, "
                        f"term={info_econ['termination_condition']})."
                    )
            except Exception as e:
                seed_record["economic"] = {
                    "status": "exception",
                    "termination_condition": "exception",
                    "exported": False,
                    "result_path": None,
                    "message": str(e),
                }
                print(f"Seed {seed}: economic baseline crashed.")
                print(f"  Error: {e}")

            # -------------------------
            # Scenario B: Sustainability-aware
            # -------------------------
            data_sust = copy.deepcopy(inst)
            data_sust["beta"] = float(data_sust.get("beta", 1.0))
            data_sust["eta"] = float(data_sust.get("eta", 1.0))
            data_sust["delta"] = float(data_sust.get("delta", 1.0))

            sust_result_path = os.path.join(
                results_dir, f"sustainability_aware_seed{seed}.json"
            )

            try:
                ok_sust, info_sust = solve_and_export(
                    data=data_sust,
                    result_path=sust_result_path,
                    solver_name=solver_name,
                    timelimit=timelimit,
                    tee=tee,
                )
                seed_record["sustainability"] = info_sust

                if ok_sust:
                    print(f"Seed {seed}: sustainability-aware saved to {sust_result_path}")
                else:
                    print(
                        f"Seed {seed}: sustainability-aware infeasible/failed "
                        f"(status={info_sust['status']}, "
                        f"term={info_sust['termination_condition']})."
                    )
            except Exception as e:
                seed_record["sustainability"] = {
                    "status": "exception",
                    "termination_condition": "exception",
                    "exported": False,
                    "result_path": None,
                    "message": str(e),
                }
                print(f"Seed {seed}: sustainability-aware crashed.")
                print(f"  Error: {e}")

            summary.append(seed_record)

        except Exception as e:
            print(f"Seed {seed}: instance generation or preprocessing failed. Continuing.")
            print(f"  Error: {e}")
            print(traceback.format_exc())

            summary.append({
                "seed": seed,
                "instance_path": None,
                "economic": {
                    "status": "not_run",
                    "termination_condition": "generation_failed",
                    "exported": False,
                    "result_path": None,
                },
                "sustainability": {
                    "status": "not_run",
                    "termination_condition": "generation_failed",
                    "exported": False,
                    "result_path": None,
                },
            })

    return summary

In [ ]:
summary = run_experiments(
    seeds=range(10, 20),
    style="clustered",
    n_bases=3,
    n_techs=12,
    n_plants=25,
    n_tickets=50,
    H=30,
    X=5,
    modes=("car", "train", "air"),
    solver_name="cplex",
    timelimit=10*60*60,
    tee=True,
)